## Distribución de Grupos - Torneo de Novatos 2026

In [3]:
## paquetes a utilizar - únicamente estos 2

import pandas as pd
import random as rd

In [ ]:
## cargar csv con los datos - lista de inscritos del torneo

lista_de_inscritos = pd.read_csv('./lista_de_inscritos.csv')

In [5]:
#### agrupar inscritos por su categoría

categorias_agrupadas = {}
for cat_nombre, grupo in lista_de_inscritos.groupby('Categoria'):
    df_temp = grupo.copy()
    df_temp['idx'] = df_temp.groupby('Club').cumcount()
    categorias_agrupadas[cat_nombre] = df_temp.pivot(index='idx', columns='Club', values='Nombre')

#display(categorias_agrupadas['Categoría D'])

In [6]:
### con esto facilito la redacción luego para mí mismo, no tiene necesariamente un propósito funcional en el código

categoria_a = categorias_agrupadas['Categoría A']
categoria_b = categorias_agrupadas['Categoría B']
categoria_c = categorias_agrupadas['Categoría C']
categoria_d = categorias_agrupadas['Categoría D']

In [ ]:
class Esgrimista:
  def __init__(self, nombre: str, club: str):
    self.nombre = nombre
    self.club = club
    self.puntaje = []

  def __repr__(self):
    return f"{self.nombre} ({self.club})"


class Grupo:
  def __init__(self, numero: int):
    self.numero = numero
    self.integrantes: list[Esgrimista] = []

  def agregar(self, esgrimista: Esgrimista):
    self.integrantes.append(esgrimista)

  def tiene_club(self, club: str) -> bool:
    return any(e.club == club for e in self.integrantes)

  def conteo_coincidencias(self) -> int:
    clubs = [e.club for e in self.integrantes]
    return len(clubs) - len(set(clubs))

  def __repr__(self):
    return f"Grupo {self.numero} (n={len(self.integrantes)}): {self.integrantes}"


class Categoria:
  def __init__(self, nombre: str):
    self.nombre = nombre
    self.esgrimistas: list[Esgrimista] = []
    self.grupos: list[Grupo] = []

  def cargar_esgrimistas(self, df_categoria: pd.DataFrame):
    for _, row in df_categoria.iterrows():
      self.esgrimistas.append(Esgrimista(row['Nombre'], row['Club']))

  def configurar_grupos(self, capacidades: list[int]):
    if sum(capacidades) != len(self.esgrimistas):
      raise ValueError(
          f"Error en {self.nombre}: La suma de cupos ({sum(capacidades)})"
          f"no coincide con el total de esgrimistas inscritos: {len(self.esgrimistas)}"
      )

    self.grupos = [Grupo(numero=i+1) for i in range(len(capacidades))]
    self._capacidades_objetivo = capacidades

  def generar_grupos(self):
    if not self.grupos:
      raise RuntimeError("Primero se debe ejecutar/llamar 'configurar:grupos()' antes de realizar la distribución de esgrimistas")
    conteo_clubes = {}
    for e in self.esgrimistas:
      conteo_clubes[e.club] = conteo_clubes.get(e.club, 0) + 1
    esgrimistas_ordenados = sorted(
        self.esgrimistas,
        key = lambda e: (conteo_clubes[e.club], rd.random()),
        reverse=True
    )

    for esgrimista in esgrimistas_ordenados:
      candidatos = [
          g for g, cap in zip(self.grupos, self._capacidades_objetivo)
          if len(g.integrantes) < cap
      ]

      sin_conflicto = [g for g in candidatos if not g.tiene_club(esgrimista.club)]
      if sin_conflicto:
        grupo_elegido = min(sin_conflicto, key=lambda g: len(g.integrantes))
      else:
        grupo_elegido = min(candidatos, key=lambda g: (g.conteo_coincidencias(), len(g.integrantes)))
      grupo_elegido.agregar(esgrimista)


  def generar_tabla_styler(self):
          max_filas = max(len(g.integrantes) for g in self.grupos)

          datos = {}
          for g in self.grupos:
              nombres = [f"{e.nombre} ({e.club})" for e in g.integrantes]
              nombres += [""] * (max_filas - len(nombres))
              datos[f"Grupo {g.numero}"] = nombres

          df_tabla = pd.DataFrame(datos)
##### comentarios respecto a la sintaxis del codigo del styler
#### caption define el titulo, en set caption se puede cambiar el texto que aparece en el titulo; allí aparece self.nombre, esto es el nombre de la categoria
### recomiendo no tocar ese trocito de self.nombre para evitar errores
## en set_table_styles va todo lo relacionado a tipo de fuente, tamaño, color, fondo y demás
##### para caption, que es el titulo encontramos color, fuente de texto, tipo de texto (normal, negrita, etc) y padding (el espacio de rellenado alrededor del elemento en la casilla)
##### lo mismo se encuentra en th, que son los encabezados de cada columna; aqupi también vemos alineación de texto, color de fondo y de texto
#### finalmente td es lo que refiere a las celdas de datos, aqupi vemos el alineamiento de texto y el padding o rellenado de los alrededores de un elemento en la celda

###### recomiendo meter los colores en formato html en lugar de buscar los nombres permitidos en la documentacion
### https://htmlcolorcodes.com/es/ esta es una buena página para obtener el html de un color
### igualmente cualquier analogo sirve
          styler = df_tabla.style.hide(axis='index') \
              .set_caption(f"Torneo de Novatos 2026 - {self.nombre}") \
              .set_table_styles([
                  {'selector': 'caption', 'props': [('color', '#1a252f'), ('font-size', '20px'), ('font-weight', 'bold'), ('padding', '10px')]},
                  {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('font-weight', 'bold'), ('text-align', 'center')]},
                  {'selector': 'td', 'props': [('text-align', 'left'), ('padding', '8px')]}
              ]) \
              .highlight_null(color='transparent')

          return styler

In [8]:
##### cantidad de inscritos por grupo
## esto es solo un indicativo para asistir en el paso siguiente
print(lista_de_inscritos['Categoria'].value_counts())

Categoria
Categoría A    22
Categoría B     8
Categoría D     7
Categoría C     7
Name: count, dtype: int64


In [11]:
## sección de configuración del torneo


####### esta parte es modificable
##### agregar el nombre respectivo de cada categoría reemplazando 'Categoría X'
##### agregar cantidad de personas en cada grupo. ej; si deseamos 3 grupos de 4 integrantes sería [4,4,4]
CONFIGURACION_TORNEO = {
    'Categoría A' : [4,4,4,5,5],
    'Categoría B' : [4,4],
    'Categoría C' : [4,3],
    'Categoría D' : [4,3]
}


####### esta parte funciona automáticamente, no requiere modificaciones (a priori) para entregar los grupos
categorias: dict[str, Categoria] = {}
tablas_estilizadas = {}

for nombre_cat, df_grupo in lista_de_inscritos.groupby('Categoria'):
    cat = Categoria(nombre_cat)
    cat.cargar_esgrimistas(df_grupo)
    estructura = CONFIGURACION_TORNEO[nombre_cat]
    cat.configurar_grupos(estructura)
    cat.generar_grupos()
    categorias[nombre_cat] = cat
    styler = cat.generar_tabla_styler()
    tablas_estilizadas[nombre_cat] = styler

    nombre_archivo = f"grupos_{nombre_cat.lower().replace(' ', '_')}.html"
    styler.to_html(nombre_archivo)

print("Épico")
############ los html quedan en la carpeta madre de este mismo notebook

Épico
